<a href="https://colab.research.google.com/github/Alcimarrfilho/llm-inference-optimization/blob/main/laboratorio_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalação
print("Instalando bibliotecas.")
# Removemos as versões fixas para que o pip escolha as melhores para o Python 3.12
!pip install -q -U transformers accelerate bitsandbytes

import torch
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Ignorar avisos desnecessários
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"

# carregamento em 4-bits para economizar VRAM
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# Resetar estatísticas da GPU para pegar o valor real
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated() / (1024 ** 2)

    print("Carregando o modelo quantizado...")
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=quantization_config,
            device_map="auto"
        )
        tokenizer = AutoTokenizer.from_pretrained(model_id)

        # Métrica exigida: Megabytes de VRAM ocupados
        mem_after = torch.cuda.max_memory_allocated() / (1024 ** 2)
        vram_utilizada = mem_after - mem_before

        print("\n" + "="*45)
        print("SUCESSO!")
        print(f"VALOR DE: {vram_utilizada:.2f} MB")
        print("="*45)
    except Exception as e:
        print(f"\n Erro ao carregar modelo: {e}")
else:
    print("Erro: GPU não detectada.")


Instalando bibliotecas.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.0 MB/s eta 0:00:00
Carregando o modelo quantizado...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]


SUCESSO!
VALOR DE: 805.93 MB


In [2]:
# Gerando o contexto fictício (aprox. 15.000 tokens)
texto_base = "O manual médico indica que o protocolo de atendimento deve ser seguido rigorosamente. "
contexto_massivo = texto_base * 700

# Tokenizando
inputs = tokenizer(contexto_massivo, return_tensors="pt").to("cuda")

# Resultado
print(f"Sucesso: Contexto gerado com {inputs.input_ids.shape[1]} tokens.")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (16102 > 2048). Running this sequence through the model will result in indexing errors


Sucesso: Contexto gerado com 16102 tokens.


In [4]:
import time

# O Problema do Decoder
print("Iniciando geração SEM otimizações. Aguarde o processamento...")

# Desativando o Cache de Memória para forçar o recalculo O(n²)
model.config.use_cache = False

# Mede tempo e memória
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    inicio = time.time()

    try:
        # geração de apenas 50 tokens
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                use_cache=False
            )

        fim = time.time()
        tempo_gargalo = fim - inicio
        memoria_pico = torch.cuda.max_memory_allocated() / (1024 ** 2)

        print(f"Resultado do Gargalo:")
        print(f"- Tempo decorrido: {tempo_gargalo:.2f} segundos")
        print(f"- Pico de VRAM: {memoria_pico:.2f} MB")

    except Exception as e:
        print(f"Erro: {e}")

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iniciando geração SEM otimizações. Aguarde o processamento...
Erro: CUDA out of memory. Tried to allocate 30.91 GiB. GPU 0 has a total capacity of 14.56 GiB of which 11.38 GiB is free. Including non-PyTorch memory, this process has 3.18 GiB memory in use. Of the allocated memory 2.59 GiB is allocated by PyTorch, and 472.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


In [6]:
import torch
import time

# Limpa os resíduos do erro anterior
torch.cuda.empty_cache()

# Ativa o KV Cache e o SDPA (otimização de hardware)
model.config.use_cache = True

print("Iniciando geração OTIMIZADA (KV Cache + Hardware-Aware)...")

torch.cuda.reset_peak_memory_stats()
inicio = time.time()

try:
    with torch.no_grad():
        # O segredo aqui é o use_cache=True
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            use_cache=True,
            do_sample=False,
            # No PyTorch moderno, o SDPA é invocado automaticamente no generate
        )

    fim = time.time()
    tempo_otimizado = fim - inicio
    memoria_pico_otimizada = torch.cuda.max_memory_allocated() / (1024 ** 2)

    print(f" SUCESSO! SISTEMA ESTABILIZADO.")
    print(f"- Tempo decorrido: {tempo_otimizado:.2f} segundos")
    print(f"- Pico de VRAM: {memoria_pico_otimizada:.2f} MB")

    # Decodificando um pedaço da resposta para provar que funcionou
    resumo = tokenizer.decode(outputs[0][-50:], skip_special_tokens=True)
    print(f"\nTrecho do Resumo Clínico Gerado: {resumo}")

except Exception as e:
    print(f"Ainda deu erro? {e}")

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iniciando geração OTIMIZADA (KV Cache + Hardware-Aware)...
Ainda deu erro? CUDA out of memory. Tried to allocate 30.91 GiB. GPU 0 has a total capacity of 14.56 GiB of which 11.50 GiB is free. Including non-PyTorch memory, this process has 3.06 GiB memory in use. Of the allocated memory 2.59 GiB is allocated by PyTorch, and 346.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


In [8]:
import torch
import gc
import time

# Limpeza melhor
if 'model_otimizado' in locals(): del model_otimizado
if 'model' in locals(): del model
gc.collect()
torch.cuda.empty_cache()

# Ajuste do Contexto para o limite da arquitetura (2048 tokens)
print("Ajustando contexto para o limite da GPU...")
input_ids = inputs['input_ids'][:, -2000:]
attention_mask = inputs['attention_mask'][:, -2000:]

# Recarregando com otimização máxima
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model_final = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    attn_implementation="sdpa"
)

# Geração Otimizada
print("Iniciando geração final otimizada...")
torch.cuda.reset_peak_memory_stats()
inicio = time.time()

with torch.no_grad():
    outputs = model_final.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=50,
        use_cache=True # KV CACHE ESSENCIAL AQUI
    )

fim = time.time()
print("\n" + "="*45)
print("SUCESSO!")
print(f"- Tempo: {fim - inicio:.2f} segundos")
print(f"- Pico de VRAM: {torch.cuda.max_memory_allocated() / (1024 ** 2):.2f} MB")
print("="*45)

Ajustando contexto para o limite da GPU...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iniciando geração final otimizada...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.



SUCESSO!
- Tempo: 4.13 segundos
- Pico de VRAM: 3055.28 MB
